# Pré-processamento 9 — Revisão de compatibilidade entre palavra e categoria
## Decisões aprovadas
Aplicar 10 correções do grupo Verbo Principal Gerúndio e 14 do grupo Nome Comun Masculino Plural e 7 do grupo Nome Comun Feminino Plural e 5 do grupo Verbo Principal Particípio Singular Masculino e 5 do grupo Verbo Principal Indicativo Passado Primeira Pessoa Singular e 7 do grupo Verbo Principal Particípio Singular Feminino e 28 do grupo Advérbio Geral e 3 do grupo Verbo Principal Indicativo Presente Primeira Pessoa Singular e 6 do grupo Adjetivo Qualificativo Masculino Singular e 7 do grupo Adjetivo Qualificativo Feminino Singular e 10 do grupo Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular e 5 do grupo Verbo Principal Subjuntivo Presente Terceira Pessoa Singular e 4 do grupo Adjetivo Qualificativo Comum Singular e 4 do grupo Verbo Principal Indicativo Passado Terceira Pessoa Plural e 7 do grupo Nome Própio Lugar e 7 do grupo Nome Comun Comum Singular e 1 do grupo Verbo Principal Particípio Plural Masculino e 1 do grupo Adjetivo Qualificativo Comum Plural e 2 do grupo Adjetivo Qualificativo Feminino Plural e 4 do grupo Nome Própio Organização (137 aprovadas ao todo). Não alterar LEMA. As únicas alterações de grafia aprovadas neste notebook são doris → dóris e minasgerais → minas gerais.
O código aceita categorias já corrigidas, permitindo reexecução. Valida palavras pela identidade e pelo lema, não pela antiga posição da linha.
Para acrescentar remoções, edite a lista e reexecute desde o carregamento.
O salvamento mostra as contagens da execução e o total final da base. Arquivos intermediários são criados em pasta temporária e limpos automaticamente ao concluir ou ocorrer uma exceção. As versões anteriores são mantidas pelo Git.


In [1]:
from pathlib import Path
import json, hashlib, os, subprocess
import pandas as pd
from IPython.display import display
arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
base_original = base.copy(deep=True)
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
print("Registros carregados:", len(base))


Registros carregados: 7688


## 1. Função de correção de categoria e preparação do resultado

In [2]:
# Função compartilhada pelos grupos; todo o código de correção está neste notebook.
# Altera exclusivamente CATMORF, sem atribuir valores em LEMA.
def aplicar_correcoes_categoria(resultado, correcoes):
    antes = resultado.copy(deep=True)
    aplicadas = []
    for item in correcoes:
        mascara = resultado["PALAVRA"].isin({item["PALAVRA"], item.get("PALAVRA_CORRIGIDA", item["PALAVRA"])})
        assert mascara.sum() == 1, f"Palavra ausente ou duplicada: {item['PALAVRA']}"
        indice = resultado.index[mascara][0]
        atual = resultado.at[indice, "CATMORF"]
        assert atual in (item["CATMORF"], item["CATEGORIA_SUGERIDA"]), f"Categoria mudou: {item['PALAVRA']}"
        assert resultado.at[indice, "LEMA"] == item["LEMA"], f"Lema mudou: {item['PALAVRA']}"
        if atual != item["CATEGORIA_SUGERIDA"]:
            resultado.at[indice, "CATMORF"] = item["CATEGORIA_SUGERIDA"]
            aplicadas.append({**item, "CATEGORIA_ANTERIOR": atual})
    pd.testing.assert_frame_equal(antes.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
    return aplicadas

resultado = base.copy(deep=True)
alteracoes = []
correcoes_aprovadas = []

# Grafias alteradas somente mediante aprovação explícita.
correcoes_grafia_aprovadas = {}
alteracoes_grafia = []


## 1.1. Verbo Principal Gerúndio — aplicar 10 correções aprovadas

In [3]:
# Lista completa das decisões aprovadas — Verbo Principal Gerúndio
correcoes_gerundio = [{'LINHA_EXCEL': 233,
  'PALAVRA': 'ajeitei',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 239,
  'PALAVRA': 'ajudar',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 296,
  'PALAVRA': 'almoçou',
  'LEMA': 'almoçar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de almoçar, terceira pessoa do singular; o gerúndio é '
                   'almoçando.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 322,
  'PALAVRA': 'amadurecer',
  'LEMA': 'amaciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 346,
  'PALAVRA': 'amassei',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 382,
  'PALAVRA': 'andarem',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Terceira Pessoa Plural',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal na terceira pessoa do plural ou futuro do subjuntivo '
                   'na mesma pessoa. A proposta de infinitivo é uma opção válida, sem preferência '
                   'obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 383,
  'PALAVRA': 'andares',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Segunda Pessoa Singular',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal ou futuro do subjuntivo na segunda pessoa do '
                   'singular; também pode ser plural do substantivo andar. A proposta verbal é uma '
                   'opção válida, sem preferência obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 426,
  'PALAVRA': 'ao',
  'LEMA': 'anunciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Preposição + Artigo Definido Masculino Singular',
  'JUSTIFICATIVA': 'Contração de a com o, compatível com o rótulo já existente na base. Não é '
                   'gerúndio de anunciar indicado no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 902,
  'PALAVRA': 'bem-vindo',
  'LEMA': 'bem-vindo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Adjetivo masculino singular; também pode ser usado em expressão de saudação. A '
                   'terminação vindo não torna o composto um gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4676,
  'PALAVRA': 'marimbondo',
  'LEMA': 'marimbondo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa o inseto; terminar em ndo não '
                   'caracteriza gerúndio por si só.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_gerundio)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_gerundio)
print("Grupo: Verbo Principal Gerúndio")
print("Correções aprovadas:", len(correcoes_gerundio))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_gerundio) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_gerundio)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Gerúndio
Correções aprovadas: 10
Aplicadas nesta execução: 0
Já estavam corrigidas: 10


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajeitei,ajeitar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
ajudar,ajudar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
almoçou,almoçar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
amadurecer,amaciar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
amassei,amassar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
andarem,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Terceira Pessoa Plural,Aprovada
andares,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Segunda Pessoa Singular,Aprovada
ao,anunciar,Verbo Principal Gerúndio,Preposição + Artigo Definido Masculino Singular,Aprovada
bem-vindo,bem-vindo,Verbo Principal Gerúndio,Adjetivo Qualificativo Masculino Singular,Aprovada
marimbondo,marimbondo,Verbo Principal Gerúndio,Nome Comun Masculino Singular,Aprovada


## 1.2. Nome Comun Masculino Plural — aplicar 14 correções aprovadas


In [4]:
# Lista completa das decisões aprovadas — Nome Comun Masculino Plural
correcoes_masculino_plural = [{'LINHA_EXCEL': 228,
  'PALAVRA': 'ainda',
  'LEMA': 'aimeudeus',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Advérbio com valores temporais, de continuidade ou inclusão; não é substantivo '
                   'masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 254,
  'PALAVRA': 'alcança',
  'LEMA': 'albuns',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Admite presente do indicativo de alcançar ou imperativo. A proposta verbal é '
                   'uma opção válida; não corresponde a nome masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 311,
  'PALAVRA': 'alça',
  'LEMA': 'albuns',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular ou forma verbal de alçar; a proposta '
                   'nominal é uma opção válida, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 315,
  'PALAVRA': 'alípio',
  'LEMA': 'alimento',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 358,
  'PALAVRA': 'americanizado',
  'LEMA': 'amendoim',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Particípio masculino singular de americanizar; também admite adjetivo. Mesmo '
                   'na leitura substantivada, a forma é singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 365,
  'PALAVRA': 'amizades',
  'LEMA': 'amigo',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; corrigir apenas o gênero da categoria.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 385,
  'PALAVRA': 'ande',
  'LEMA': 'andar',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de andar no presente do subjuntivo; primeira pessoa do singular ou '
                   'imperativo também são possíveis. A análise proposta não tem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 393,
  'PALAVRA': 'ângelo',
  'LEMA': 'anel',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 404,
  'PALAVRA': 'anotar',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de anotar. Uma eventual leitura nominal do infinitivo seria '
                   'singular; não é nome masculino plural na forma cadastrada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 406,
  'PALAVRA': 'ansiosa',
  'LEMA': 'ano',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Forma feminina singular de ansioso. Mesmo em leitura substantivada, o gênero e '
                   'o número da categoria atual não correspondem.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2450,
  'PALAVRA': 'dicas',
  'LEMA': 'dica',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Substantivo feminino no plural; corrigir apenas o gênero da categoria.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6526,
  'PALAVRA': 'rubens',
  'LEMA': 'rubens',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Alípio, Ângelo ou Rubens, respectivamente; '
                   'não corresponde à classificação de nome comum masculino plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7135,
  'PALAVRA': 'teus',
  'LEMA': 'teus',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Determinante Possessivo Segunda Pessoa Masculino Plural Singular',
  'JUSTIFICATIVA': 'Possessivo da segunda pessoa: masculino plural para o elemento possuído, '
                   'possuidor singular. Também admite análise pronominal. O rótulo proposto é novo '
                   'na base e segue o padrão de teu e meus, sem alterar LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7246,
  'PALAVRA': 'torres',
  'LEMA': 'torr',
  'CATMORF': 'Nome Comun Masculino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Plural feminino de torre; também admite sobrenome ou segunda pessoa do '
                   'presente de torrar. A proposta nominal feminina é uma análise possível.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_masculino_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_masculino_plural)
print("Grupo: Nome Comun Masculino Plural")
print("Correções aprovadas:", len(correcoes_masculino_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_masculino_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_masculino_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Masculino Plural
Correções aprovadas: 14
Aplicadas nesta execução: 0
Já estavam corrigidas: 14


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ainda,aimeudeus,Nome Comun Masculino Plural,Advérbio Geral,Aprovada
alcança,albuns,Nome Comun Masculino Plural,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
alça,albuns,Nome Comun Masculino Plural,Nome Comun Feminino Singular,Aprovada
alípio,alimento,Nome Comun Masculino Plural,Nome Própio Pessoa,Aprovada
americanizado,amendoim,Nome Comun Masculino Plural,Verbo Principal Particípio Singular Masculino,Aprovada
amizades,amigo,Nome Comun Masculino Plural,Nome Comun Feminino Plural,Aprovada
ande,andar,Nome Comun Masculino Plural,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
ângelo,anel,Nome Comun Masculino Plural,Nome Própio Pessoa,Aprovada
anotar,ano,Nome Comun Masculino Plural,Verbo Principal Infinitivo,Aprovada
ansiosa,ano,Nome Comun Masculino Plural,Adjetivo Qualificativo Feminino Singular,Aprovada


## 1.3. Nome Comun Feminino Plural — aplicar 7 correções aprovadas


In [5]:
# Lista completa das decisões aprovadas — Nome Comun Feminino Plural
correcoes_feminino_plural = [{'LINHA_EXCEL': 260,
  'PALAVRA': 'álcool',
  'LEMA': 'alça',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; gênero e número não correspondem à categoria '
                   'feminina plural. Preservar o lema cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 363,
  'PALAVRA': 'amigos',
  'LEMA': 'amígdala',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural masculino de amigo; corrigir o gênero da categoria, sem alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 368,
  'PALAVRA': 'amora',
  'LEMA': 'amizade',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular, nome da fruta; também pode ser forma de '
                   'amorar. A proposta nominal é uma análise possível; a forma não é plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 419,
  'PALAVRA': 'antídoto',
  'LEMA': 'antena',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular; gênero e número não correspondem à categoria '
                   'feminina plural. Preservar o lema cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2055,
  'PALAVRA': 'crianções',
  'LEMA': 'criança',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural de crianção, forma aumentativa masculina. Corrigir o gênero da '
                   'categoria, preservando o lema criança e a grafia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6183,
  'PALAVRA': 'quebra-cabeças',
  'LEMA': 'quebra-cabeças',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'O composto é substantivo masculino de dois números. A leitura plural é '
                   'possível; corrigir apenas o gênero, sem impor preferência entre singular e '
                   'plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6621,
  'PALAVRA': 'salvá-las',
  'LEMA': 'salvá-las',
  'CATMORF': 'Nome Comun Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de salvar com pronome enclítico las. O gênero e o número do pronome '
                   'não tornam a expressão um substantivo feminino plural.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_feminino_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_feminino_plural)
print("Grupo: Nome Comun Feminino Plural")
print("Correções aprovadas:", len(correcoes_feminino_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_feminino_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_feminino_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Feminino Plural
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
álcool,alça,Nome Comun Feminino Plural,Nome Comun Masculino Singular,Aprovada
amigos,amígdala,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
amora,amizade,Nome Comun Feminino Plural,Nome Comun Feminino Singular,Aprovada
antídoto,antena,Nome Comun Feminino Plural,Nome Comun Masculino Singular,Aprovada
crianções,criança,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
quebra-cabeças,quebra-cabeças,Nome Comun Feminino Plural,Nome Comun Masculino Plural,Aprovada
salvá-las,salvá-las,Nome Comun Feminino Plural,Verbo Principal Infinitivo,Aprovada


## 1.4. Verbo Principal Particípio Singular Masculino — aplicar 5 correções aprovadas


In [6]:
# Lista completa das decisões aprovadas — Verbo Principal Particípio Singular Masculino
correcoes_participio_masculino = [{'LINHA_EXCEL': 250,
  'PALAVRA': 'alaranjadas',
  'LEMA': 'alar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Plural Feminino',
  'JUSTIFICATIVA': 'Particípio feminino plural de alaranjar; também admite adjetivo. A leitura de '
                   'particípio pode ser mantida, corrigindo gênero e número, sem alterar o lema '
                   'cadastrado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 336,
  'PALAVRA': 'amarre',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Presente do subjuntivo de amarrar; primeira pessoa do singular ou imperativo '
                   'também são possíveis. A proposta é uma análise válida, sem preferência '
                   'obrigatória; não é particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 345,
  'PALAVRA': 'amassasse',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Imperfeito Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito imperfeito do subjuntivo de amassar. Primeira e terceira pessoas do '
                   'singular são possíveis; a proposta não estabelece preferência. Não é '
                   'particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 397,
  'PALAVRA': 'animal',
  'LEMA': 'animar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular; também pode ser adjetivo de dois '
                   'gêneros. Não é particípio de animar, cujo particípio é animado.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 412,
  'PALAVRA': 'anterior',
  'LEMA': 'antecipar',
  'CATMORF': 'Verbo Principal Particípio Singular Masculino',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Adjetivo de dois gêneros no singular, para aquilo que vem antes. Não é '
                   'particípio de antecipar, cujo particípio é antecipado.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_participio_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_participio_masculino)
print("Grupo: Verbo Principal Particípio Singular Masculino")
print("Correções aprovadas:", len(correcoes_participio_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_participio_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_participio_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Particípio Singular Masculino
Correções aprovadas: 5
Aplicadas nesta execução: 0
Já estavam corrigidas: 5


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
alaranjadas,alar,Verbo Principal Particípio Singular Masculino,Verbo Principal Particípio Plural Feminino,Aprovada
amarre,amarrar,Verbo Principal Particípio Singular Masculino,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
amassasse,amassar,Verbo Principal Particípio Singular Masculino,Verbo Principal Subjuntivo Imperfeito Terceira Pessoa Singular,Aprovada
animal,animar,Verbo Principal Particípio Singular Masculino,Nome Comun Masculino Singular,Aprovada
anterior,antecipar,Verbo Principal Particípio Singular Masculino,Adjetivo Qualificativo Comum Singular,Aprovada


## 1.5. Verbo Principal Indicativo Passado Primeira Pessoa Singular — aplicar 5 correções aprovadas


In [7]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Passado Primeira Pessoa Singular
correcoes_passado_primeira = [{'LINHA_EXCEL': 245,
  'PALAVRA': 'ajudou',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de ajudar na terceira pessoa do singular; a primeira pessoa '
                   'é ajudei.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 348,
  'PALAVRA': 'ambiente',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular, além de adjetivo. A proposta nominal é '
                   'uma opção válida; não corresponde a forma do passado de amassar indicada no '
                   'lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 389,
  'PALAVRA': 'andreia',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Andreia, preservando a grafia cadastrada; não '
                   'corresponde à primeira pessoa do passado de andar indicada no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2418,
  'PALAVRA': 'detetive',
  'LEMA': 'detetive',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo que designa investigador; a leitura masculina singular é possível, '
                   'sem preferência obrigatória sobre outras análises nominais compatíveis. Não é '
                   'forma de passado na primeira pessoa.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5779,
  'PALAVRA': 'piuí',
  'LEMA': 'piuí',
  'CATMORF': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino que designa aves chamadas piuí. A proposta segue '
                   'essa leitura lexical; se a ocorrência for apenas vocalização, reavaliar a '
                   'categoria com contexto.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_passado_primeira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_passado_primeira)
print("Grupo: Verbo Principal Indicativo Passado Primeira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_passado_primeira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_passado_primeira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_passado_primeira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Passado Primeira Pessoa Singular
Correções aprovadas: 5
Aplicadas nesta execução: 0
Já estavam corrigidas: 5


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajudou,ajudar,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
ambiente,amassar,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Nome Comun Masculino Singular,Aprovada
andreia,andar,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Nome Própio Pessoa,Aprovada
detetive,detetive,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Nome Comun Masculino Singular,Aprovada
piuí,piuí,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Nome Comun Masculino Singular,Aprovada


## 1.6. Verbo Principal Particípio Singular Feminino — aplicar 7 correções aprovadas


In [8]:
# Lista completa das decisões aprovadas — Verbo Principal Particípio Singular Feminino
correcoes_participio_feminino = [{'LINHA_EXCEL': 232,
  'PALAVRA': 'ajeitar',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de ajeitar; não é particípio. A forma participial feminina seria '
                   'ajeitada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 272,
  'PALAVRA': 'alface',
  'LEMA': 'alfabetizar',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo feminino singular que designa a hortaliça; não é particípio de '
                   'alfabetizar indicado no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 321,
  'PALAVRA': 'amadurece',
  'LEMA': 'amaciar',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Presente de amadurecer; também admite imperativo. A proposta é uma análise '
                   'possível, sem preferência obrigatória; não é particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 335,
  'PALAVRA': 'amarraram',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'JUSTIFICATIVA': 'Forma de amarrar na terceira pessoa do plural do passado, compatível com '
                   'pretérito perfeito ou mais-que-perfeito. Não é particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 343,
  'PALAVRA': 'amassando',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de amassar; o particípio feminino é amassada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2669,
  'PALAVRA': 'eita',
  'LEMA': 'eita',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Interjeição',
  'JUSTIFICATIVA': 'Expressão exclamativa de satisfação, espanto ou surpresa; não é particípio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6944,
  'PALAVRA': 'sujeirada',
  'LEMA': 'sujeirada',
  'CATMORF': 'Verbo Principal Particípio Singular Feminino',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo feminino derivado de sujeira com o sufixo ada, registrado em '
                   'dicionário. A terminação ada, por si só, não caracteriza particípio.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_participio_feminino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_participio_feminino)
print("Grupo: Verbo Principal Particípio Singular Feminino")
print("Correções aprovadas:", len(correcoes_participio_feminino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_participio_feminino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_participio_feminino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Particípio Singular Feminino
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajeitar,ajeitar,Verbo Principal Particípio Singular Feminino,Verbo Principal Infinitivo,Aprovada
alface,alfabetizar,Verbo Principal Particípio Singular Feminino,Nome Comun Feminino Singular,Aprovada
amadurece,amaciar,Verbo Principal Particípio Singular Feminino,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
amarraram,amarrar,Verbo Principal Particípio Singular Feminino,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Aprovada
amassando,amassar,Verbo Principal Particípio Singular Feminino,Verbo Principal Gerúndio,Aprovada
eita,eita,Verbo Principal Particípio Singular Feminino,Interjeição,Aprovada
sujeirada,sujeirada,Verbo Principal Particípio Singular Feminino,Nome Comun Feminino Singular,Aprovada


## 1.7. Advérbio Geral — aplicar 28 correções aprovadas


In [9]:
# Lista completa das decisões aprovadas — Advérbio Geral
correcoes_adverbio_geral = [{'LINHA_EXCEL': 229,
  'PALAVRA': 'ajeita',
  'LEMA': 'ainda',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de presente do indicativo, respectivamente de ajeitar e alimentar; '
                   'também admite imperativo. A proposta é uma análise possível, sem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 230,
  'PALAVRA': 'ajeitada',
  'LEMA': 'aió',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Feminino',
  'JUSTIFICATIVA': 'Particípio feminino singular de ajeitar; também admite leitura adjetival ou '
                   'nominal, sem preferência obrigatória. Não alterar o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 281,
  'PALAVRA': 'alice',
  'LEMA': 'ali',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal; não inferir a classe pelo lema cadastrado, '
                   'que diverge da palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 282,
  'PALAVRA': 'alimenta',
  'LEMA': 'aliás',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Forma de presente do indicativo, respectivamente de ajeitar e alimentar; '
                   'também admite imperativo. A proposta é uma análise possível, sem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 286,
  'PALAVRA': 'alison',
  'LEMA': 'alinã',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal; não inferir a classe pelo lema cadastrado, '
                   'que diverge da palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 298,
  'PALAVRA': 'alta',
  'LEMA': 'aló',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival feminina singular (liinda como alongamento de linda). '
                   'Não corresponde a advérbio na leitura lexical proposta; também pode haver uso '
                   'nominal.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 299,
  'PALAVRA': 'altas',
  'LEMA': 'aló',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Plural',
  'JUSTIFICATIVA': 'Admite leitura adjetival feminina plural; também há leitura nominal. A '
                   'proposta adjetival é uma opção possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 305,
  'PALAVRA': 'alugo',
  'LEMA': 'alto',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Primeira pessoa do presente de alugar, não advérbio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 314,
  'PALAVRA': 'alícia',
  'LEMA': 'ali',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal; não inferir a classe pelo lema cadastrado, '
                   'que diverge da palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 323,
  'PALAVRA': 'amanheceu',
  'LEMA': 'amanhã',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de amanhecer na terceira pessoa do singular, não advérbio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 325,
  'PALAVRA': 'amar',
  'LEMA': 'amanhã',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Admite infinitivo verbal. Anular também pode ser adjetivo ou substantivo; a '
                   'proposta verbal é uma opção possível, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 326,
  'PALAVRA': 'amarela',
  'LEMA': 'amanhã',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival feminina singular (liinda como alongamento de linda). '
                   'Não corresponde a advérbio na leitura lexical proposta; também pode haver uso '
                   'nominal.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 411,
  'PALAVRA': 'antenas',
  'LEMA': 'antecipadamente',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Plural',
  'JUSTIFICATIVA': 'Plural feminino de antena; não corresponde a advérbio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 420,
  'PALAVRA': 'anular',
  'LEMA': 'antigamente',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Admite infinitivo verbal. Anular também pode ser adjetivo ou substantivo; a '
                   'proposta verbal é uma opção possível, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 428,
  'PALAVRA': 'aos',
  'LEMA': 'aonde',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Preposição + Artigo Definido Masculino Plural',
  'JUSTIFICATIVA': 'Contração de a com os, não advérbio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1034,
  'PALAVRA': 'borocoxô',
  'LEMA': 'borocoxô',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Adjetivo de dois gêneros; também admite substantivo. A proposta considera a '
                   'leitura lexical adjetival; se houver emprego adverbial contextual comprovado, '
                   'manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1328,
  'PALAVRA': 'carijó',
  'LEMA': 'carijó',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Adjetivo de dois gêneros; também admite substantivo. A proposta considera a '
                   'leitura lexical adjetival; se houver emprego adverbial contextual comprovado, '
                   'manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1404,
  'PALAVRA': 'catapora',
  'LEMA': 'catapora',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular: a doença ou a designação afetiva de '
                   'madrinha, respectivamente. Se dinda nomear uma personagem, nome próprio também '
                   'é possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2470,
  'PALAVRA': 'dinda',
  'LEMA': 'dinda',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular: a doença ou a designação afetiva de '
                   'madrinha, respectivamente. Se dinda nomear uma personagem, nome próprio também '
                   'é possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2665,
  'PALAVRA': 'egito',
  'LEMA': 'egito',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Própio Lugar',
  'JUSTIFICATIVA': 'Admite leitura como nome geográfico Egito; preservar a caixa e o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2842,
  'PALAVRA': 'enquanto',
  'LEMA': 'enquanto',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Conjunção Subordinada',
  'JUSTIFICATIVA': 'Admite conjunção temporal ou de contraste. Por enquanto é locução adverbial, '
                   'mas isso não torna enquanto isoladamente advérbio na análise lexical proposta.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3093,
  'PALAVRA': 'espinhudo',
  'LEMA': 'espinhudo',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival masculina singular. A proposta considera essa leitura '
                   'lexical; se houver emprego adverbial contextual comprovado, manter a categoria '
                   'atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4862,
  'PALAVRA': 'mixuruco',
  'LEMA': 'mixuruco',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura adjetival masculina singular. A proposta considera essa leitura '
                   'lexical; se houver emprego adverbial contextual comprovado, manter a categoria '
                   'atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5125,
  'PALAVRA': 'neném',
  'LEMA': 'neném',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura nominal masculina singular: hipopótamo, lápis, monstro, neném, '
                   'piuí ou robô, respectivamente. As interpretações de grafias não convencionais '
                   'ficam sujeitas à confirmação; preservar grafia e lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6166,
  'PALAVRA': 'quanta',
  'LEMA': 'quanta',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Pronome Interrogativo Feminino Singular Indefinido',
  'JUSTIFICATIVA': 'Admite pronome/quantificador interrogativo feminino singular; também há '
                   'leitura exclamativa ou nominal (plural de quantum). A proposta pronominal é '
                   'uma opção possível. Este rótulo é novo na base e segue o padrão de quantas; '
                   'não alterar LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6473,
  'PALAVRA': 'robô',
  'LEMA': 'robô',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite leitura nominal masculina singular: hipopótamo, lápis, monstro, neném, '
                   'piuí ou robô, respectivamente. As interpretações de grafias não convencionais '
                   'ficam sujeitas à confirmação; preservar grafia e lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6618,
  'PALAVRA': 'salvo',
  'LEMA': 'salvo',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Masculino',
  'JUSTIFICATIVA': 'Admite particípio masculino singular de salvar; também pode ser adjetivo, '
                   'primeira pessoa do presente ou preposição. A salvo é locução adverbial, mas '
                   'salvo isolado admite as outras classes indicadas; a proposta participial é uma '
                   'opção possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7381,
  'PALAVRA': 'três',
  'LEMA': 'três',
  'CATMORF': 'Advérbio Geral',
  'CATEGORIA_SUGERIDA': 'Quantidade',
  'JUSTIFICATIVA': 'Numeral cardinal; usar a classificação Quantidade já adotada para numerais. '
                   'Também admite leitura nominal, sem preferência obrigatória.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adverbio_geral)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adverbio_geral)
print("Grupo: Advérbio Geral")
print("Correções aprovadas:", len(correcoes_adverbio_geral))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adverbio_geral) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adverbio_geral)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Advérbio Geral
Correções aprovadas: 28
Aplicadas nesta execução: 0
Já estavam corrigidas: 28


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajeita,ainda,Advérbio Geral,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
ajeitada,aió,Advérbio Geral,Verbo Principal Particípio Singular Feminino,Aprovada
alice,ali,Advérbio Geral,Nome Própio Pessoa,Aprovada
alimenta,aliás,Advérbio Geral,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
alison,alinã,Advérbio Geral,Nome Própio Pessoa,Aprovada
alta,aló,Advérbio Geral,Adjetivo Qualificativo Feminino Singular,Aprovada
altas,aló,Advérbio Geral,Adjetivo Qualificativo Feminino Plural,Aprovada
alugo,alto,Advérbio Geral,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Aprovada
alícia,ali,Advérbio Geral,Nome Própio Pessoa,Aprovada
amanheceu,amanhã,Advérbio Geral,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada


## 1.8. Verbo Principal Indicativo Presente Primeira Pessoa Singular — aplicar 3 correções aprovadas

In [10]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Presente Primeira Pessoa Singular
correcoes_presente_primeira = [{'LINHA_EXCEL': 328,
  'PALAVRA': 'amarrada',
  'LEMA': 'amargar',
  'CATMORF': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Feminino',
  'JUSTIFICATIVA': 'Forma feminina singular do particípio de amarrar; também admite uso adjetival. '
                   'Não corresponde à primeira pessoa do presente.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 336,
  'PALAVRA': 'amassada',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Feminino',
  'JUSTIFICATIVA': 'Forma feminina singular do particípio de amassar; também admite uso adjetival. '
                   'Não corresponde à primeira pessoa do presente.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5532,
  'PALAVRA': 'peixe-sapo',
  'LEMA': 'peixe-sapo',
  'CATMORF': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo composto masculino singular que designa um peixe; não corresponde '
                   'a uma forma verbal na primeira pessoa.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_presente_primeira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_presente_primeira)
print("Grupo: Verbo Principal Indicativo Presente Primeira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_presente_primeira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_presente_primeira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_presente_primeira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Presente Primeira Pessoa Singular
Correções aprovadas: 3
Aplicadas nesta execução: 0
Já estavam corrigidas: 3


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
amarrada,amargar,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Verbo Principal Particípio Singular Feminino,Aprovada
amassada,amarrar,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Verbo Principal Particípio Singular Feminino,Aprovada
peixe-sapo,peixe-sapo,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Nome Comun Masculino Singular,Aprovada


## 1.9. Adjetivo Qualificativo Masculino Singular — aplicar 6 correções aprovadas

In [11]:
# Lista completa das decisões aprovadas — Adjetivo Qualificativo Masculino Singular
correcoes_adjetivo_masculino = [{'LINHA_EXCEL': 418,
  'PALAVRA': 'anunciar',
  'LEMA': 'antigo',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo do verbo anunciar; não corresponde a adjetivo masculino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2212,
  'PALAVRA': 'denovo',
  'LEMA': 'denovo',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Advérbio Geral',
  'JUSTIFICATIVA': 'Interpretado como de novo, no sentido de novamente ou outra vez: locução de '
                   'valor adverbial. Usar o rótulo Advérbio Geral disponível na base, preservando '
                   'a grafia e o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 2918,
  'PALAVRA': 'ernesto',
  'LEMA': 'ernesto',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como o nome pessoal Ernesto, não como adjetivo qualificativo. '
                   'Preservar a caixa e o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3667,
  'PALAVRA': 'fólego',
  'LEMA': 'fólego',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Interpretado como fôlego, designa respiração, capacidade ou ânimo; leitura '
                   'nominal masculina singular. Preservar a grafia e o lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5703,
  'PALAVRA': 'pinico',
  'LEMA': 'pinico',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Pinico é substantivo masculino (ponta aguda, bico); se representar penico, a '
                   'leitura também é nominal masculina. Também admite forma de pinicar; a proposta '
                   'nominal é uma opção possível.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6139,
  'PALAVRA': 'quadro-negro',
  'LEMA': 'quadro-negro',
  'CATMORF': 'Adjetivo Qualificativo Masculino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo composto masculino singular que designa o quadro usado para '
                   'escrita; negro integra o composto e não torna a palavra inteira adjetivo.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adjetivo_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adjetivo_masculino)
print("Grupo: Adjetivo Qualificativo Masculino Singular")
print("Correções aprovadas:", len(correcoes_adjetivo_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adjetivo_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adjetivo_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Adjetivo Qualificativo Masculino Singular
Correções aprovadas: 6
Aplicadas nesta execução: 0
Já estavam corrigidas: 6


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
anunciar,antigo,Adjetivo Qualificativo Masculino Singular,Verbo Principal Infinitivo,Aprovada
denovo,denovo,Adjetivo Qualificativo Masculino Singular,Advérbio Geral,Aprovada
ernesto,ernesto,Adjetivo Qualificativo Masculino Singular,Nome Própio Pessoa,Aprovada
fólego,fólego,Adjetivo Qualificativo Masculino Singular,Nome Comun Masculino Singular,Aprovada
pinico,pinico,Adjetivo Qualificativo Masculino Singular,Nome Comun Masculino Singular,Aprovada
quadro-negro,quadro-negro,Adjetivo Qualificativo Masculino Singular,Nome Comun Masculino Singular,Aprovada


## 1.10. Adjetivo Qualificativo Feminino Singular — aplicar 7 correções aprovadas

In [12]:
# Lista completa das decisões aprovadas — Adjetivo Qualificativo Feminino Singular
correcoes_adjetivo_feminino = [{'LINHA_EXCEL': 161,
  'PALAVRA': 'adriana',
  'LEMA': 'adriana',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Adriana. A proposta considera o antropônimo, '
                   'preservando caixa e LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 413,
  'PALAVRA': 'antigo',
  'LEMA': 'antigo',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Adjetivo na forma masculina singular; a forma feminina é antiga. Corrigir '
                   'apenas a categoria, mantendo a palavra e o LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3645,
  'PALAVRA': 'furadeira',
  'LEMA': 'furadeira',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo feminino que designa a ferramenta usada para fazer furos. A '
                   'proposta considera essa leitura lexical; se houver uso adjetival contextual '
                   'comprovado, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4140,
  'PALAVRA': 'isadora',
  'LEMA': 'isadora',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite leitura como nome pessoal Isadora, não como adjetivo qualificativo. '
                   'Preservar caixa e LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4335,
  'PALAVRA': 'lembrancinha',
  'LEMA': 'lembrancinha',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo nominal de lembrança, que pode designar pequeno presente ou '
                   'recordação; feminino singular.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5621,
  'PALAVRA': 'pernona',
  'LEMA': 'pernão',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Aumentativo nominal feminino de perna: perna grande ou grossa. Não confundir '
                   'com o adjetivo pernão, no sentido de ímpar; usar o rótulo nominal feminino '
                   'singular existente na base.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5877,
  'PALAVRA': 'pracinha',
  'LEMA': 'pracinha',
  'CATMORF': 'Adjetivo Qualificativo Feminino Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular como diminutivo de praça. Também existe a '
                   'leitura militar, de outro gênero; a proposta feminina é uma análise possível, '
                   'sem preferência obrigatória.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adjetivo_feminino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adjetivo_feminino)
print("Grupo: Adjetivo Qualificativo Feminino Singular")
print("Correções aprovadas:", len(correcoes_adjetivo_feminino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adjetivo_feminino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adjetivo_feminino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Adjetivo Qualificativo Feminino Singular
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
adriana,adriana,Adjetivo Qualificativo Feminino Singular,Nome Própio Pessoa,Aprovada
antigo,antigo,Adjetivo Qualificativo Feminino Singular,Adjetivo Qualificativo Masculino Singular,Aprovada
furadeira,furadeira,Adjetivo Qualificativo Feminino Singular,Nome Comun Feminino Singular,Aprovada
isadora,isadora,Adjetivo Qualificativo Feminino Singular,Nome Própio Pessoa,Aprovada
lembrancinha,lembrancinha,Adjetivo Qualificativo Feminino Singular,Nome Comun Feminino Singular,Aprovada
pernona,pernão,Adjetivo Qualificativo Feminino Singular,Nome Comun Feminino Singular,Aprovada
pracinha,pracinha,Adjetivo Qualificativo Feminino Singular,Nome Comun Feminino Singular,Aprovada


## 1.11. Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular — aplicar 10 correções aprovadas

In [13]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular
correcoes_imperfeito_terceira = [{'LINHA_EXCEL': 382,
  'PALAVRA': 'andei',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de andar na primeira pessoa singular, não terceira pessoa '
                   'do imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 965,
  'PALAVRA': 'bobinha',
  'LEMA': 'bobinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo feminino singular de boba; admite adjetivo, não a flexão verbal '
                   'atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3311,
  'PALAVRA': 'farofinha',
  'LEMA': 'farofinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo nominal feminino de farofa; não é forma verbal no imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3421,
  'PALAVRA': 'fichinha',
  'LEMA': 'fichinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular, como diminutivo de ficha; não é forma '
                   'verbal no imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3441,
  'PALAVRA': 'filhotinha',
  'LEMA': 'filhotinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo nominal feminino de filhota; não é forma verbal no imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3753,
  'PALAVRA': 'girafinha',
  'LEMA': 'girafinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo nominal feminino de girafa; não é forma verbal no imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3979,
  'PALAVRA': 'ibirapuera',
  'LEMA': 'ibirapuera',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Lugar',
  'JUSTIFICATIVA': 'Admite nome geográfico, como o Parque Ibirapuera, não forma verbal.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4374,
  'PALAVRA': 'levinha',
  'LEMA': 'levinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Diminutivo feminino singular de leve; admite adjetivo, não forma verbal no '
                   'imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6566,
  'PALAVRA': 'salsinha',
  'LEMA': 'salsinha',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Substantivo feminino que designa a erva usada como tempero; não é forma verbal '
                   'no imperfeito.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7136,
  'PALAVRA': 'titia',
  'LEMA': 'titia',
  'CATMORF': 'Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Designação nominal afetiva de tia; não é forma verbal no imperfeito.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_imperfeito_terceira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_imperfeito_terceira)
print("Grupo: Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_imperfeito_terceira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_imperfeito_terceira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_imperfeito_terceira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular
Correções aprovadas: 10
Aplicadas nesta execução: 0
Já estavam corrigidas: 10


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
andei,andar,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Verbo Principal Indicativo Passado Primeira Pessoa Singular,Aprovada
bobinha,bobinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Adjetivo Qualificativo Feminino Singular,Aprovada
farofinha,farofinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
fichinha,fichinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
filhotinha,filhotinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
girafinha,girafinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
ibirapuera,ibirapuera,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Própio Lugar,Aprovada
levinha,levinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Adjetivo Qualificativo Feminino Singular,Aprovada
salsinha,salsinha,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
titia,titia,Verbo Principal Indicativo Imperfeito Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada


## 1.12. Verbo Principal Subjuntivo Presente Terceira Pessoa Singular — aplicar 5 correções aprovadas

In [14]:
# Lista completa das decisões aprovadas — Verbo Principal Subjuntivo Presente Terceira Pessoa Singular
correcoes_subjuntivo_presente_terceira = [{'LINHA_EXCEL': 241,
  'PALAVRA': 'ajudo',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Primeira pessoa singular do presente do indicativo de ajudar; no subjuntivo '
                   'seria ajude.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 261,
  'PALAVRA': 'alegres',
  'LEMA': 'alegrar',
  'CATMORF': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Segunda Pessoa Singular',
  'JUSTIFICATIVA': 'Admite que tu alegres, de alegrar: segunda pessoa singular. Também admite '
                   'adjetivo plural; a proposta verbal é uma análise possível, sem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 262,
  'PALAVRA': 'alegria',
  'LEMA': 'alegrar',
  'CATMORF': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo feminino singular que designa estado de contentamento; não '
                   'é terceira pessoa do presente do subjuntivo de alegrar.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 335,
  'PALAVRA': 'amassa',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Admite terceira pessoa singular do presente do indicativo de amassar, além de '
                   'imperativo; no subjuntivo seria amasse.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 384,
  'PALAVRA': 'andrea',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite nome pessoal Andrea ou Andréa. A proposta considera o antropônimo; '
                   'preservar grafia e LEMA, sem inferir classe a partir do lema andar.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_subjuntivo_presente_terceira)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_subjuntivo_presente_terceira)
print("Grupo: Verbo Principal Subjuntivo Presente Terceira Pessoa Singular")
print("Correções aprovadas:", len(correcoes_subjuntivo_presente_terceira))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_subjuntivo_presente_terceira) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_subjuntivo_presente_terceira)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Subjuntivo Presente Terceira Pessoa Singular
Correções aprovadas: 5
Aplicadas nesta execução: 0
Já estavam corrigidas: 5


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajudo,ajudar,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Verbo Principal Indicativo Presente Primeira Pessoa Singular,Aprovada
alegres,alegrar,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Verbo Principal Subjuntivo Presente Segunda Pessoa Singular,Aprovada
alegria,alegrar,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Nome Comun Feminino Singular,Aprovada
amassa,amarrar,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Verbo Principal Indicativo Presente Terceira Pessoa Singular,Aprovada
andrea,andar,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Nome Própio Pessoa,Aprovada


## 1.13. Adjetivo Qualificativo Comum Singular — aplicar 4 correções aprovadas

In [15]:
# Lista completa das decisões aprovadas — Adjetivo Qualificativo Comum Singular
correcoes_adjetivo_comum_singular = [{'LINHA_EXCEL': 346,
  'PALAVRA': 'ame',
  'LEMA': 'amável',
  'CATMORF': 'Adjetivo Qualificativo Comum Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Admite que ele ame, presente do subjuntivo de amar, além de imperativo. Não '
                   'inferir a classe pelo lema amável.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 394,
  'PALAVRA': 'animou',
  'LEMA': 'animal',
  'CATMORF': 'Adjetivo Qualificativo Comum Singular',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Terceira pessoa singular do pretérito perfeito de animar; não adjetivo. '
                   'Preservar o lema animal.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 410,
  'PALAVRA': 'antiga',
  'LEMA': 'anterior',
  'CATMORF': 'Adjetivo Qualificativo Comum Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Forma adjetival feminina singular de antigo; não é adjetivo comum aos dois '
                   'gêneros.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 1178,
  'PALAVRA': 'cachorro-quente',
  'LEMA': 'cachorro-quente',
  'CATMORF': 'Adjetivo Qualificativo Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo composto masculino singular que designa o alimento; quente integra '
                   'o composto e não torna o conjunto adjetivo.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adjetivo_comum_singular)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adjetivo_comum_singular)
print("Grupo: Adjetivo Qualificativo Comum Singular")
print("Correções aprovadas:", len(correcoes_adjetivo_comum_singular))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adjetivo_comum_singular) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adjetivo_comum_singular)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Adjetivo Qualificativo Comum Singular
Correções aprovadas: 4
Aplicadas nesta execução: 0
Já estavam corrigidas: 4


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ame,amável,Adjetivo Qualificativo Comum Singular,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
animou,animal,Adjetivo Qualificativo Comum Singular,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada
antiga,anterior,Adjetivo Qualificativo Comum Singular,Adjetivo Qualificativo Feminino Singular,Aprovada
cachorro-quente,cachorro-quente,Adjetivo Qualificativo Comum Singular,Nome Comun Masculino Singular,Aprovada


## 1.14. Verbo Principal Indicativo Passado Terceira Pessoa Plural — aplicar 4 correções aprovadas

In [16]:
# Lista completa das decisões aprovadas — Verbo Principal Indicativo Passado Terceira Pessoa Plural
correcoes_passado_terceira_plural = [{'LINHA_EXCEL': 239,
  'PALAVRA': 'ajude',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Subjuntivo Presente Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Admite presente do subjuntivo de ajudar, na terceira pessoa singular, além de '
                   'outras leituras verbais; não é passado na terceira pessoa plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 314,
  'PALAVRA': 'amaciada',
  'LEMA': 'am',
  'CATMORF': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Particípio Singular Feminino',
  'JUSTIFICATIVA': 'Particípio feminino singular de amaciar, também usado como adjetivo; não é '
                   'passado na terceira pessoa plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 315,
  'PALAVRA': 'amaciando',
  'LEMA': 'am',
  'CATMORF': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de amaciar, não passado na terceira pessoa plural.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 334,
  'PALAVRA': 'amarrou',
  'LEMA': 'amarrar',
  'CATMORF': 'Verbo Principal Indicativo Passado Terceira Pessoa Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de amarrar na terceira pessoa singular; o plural seria '
                   'amarraram.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_passado_terceira_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_passado_terceira_plural)
print("Grupo: Verbo Principal Indicativo Passado Terceira Pessoa Plural")
print("Correções aprovadas:", len(correcoes_passado_terceira_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_passado_terceira_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_passado_terceira_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Indicativo Passado Terceira Pessoa Plural
Correções aprovadas: 4
Aplicadas nesta execução: 0
Já estavam corrigidas: 4


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajude,ajudar,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Verbo Principal Subjuntivo Presente Terceira Pessoa Singular,Aprovada
amaciada,am,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Verbo Principal Particípio Singular Feminino,Aprovada
amaciando,am,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Verbo Principal Gerúndio,Aprovada
amarrou,amarrar,Verbo Principal Indicativo Passado Terceira Pessoa Plural,Verbo Principal Indicativo Passado Terceira Pessoa Singular,Aprovada


## 1.15. Verbo Principal Indicativo Presente Terceira Pessoa Plural — remoção aprovada

Remover otrem pela lista da seção 2. Não há correções de categoria neste grupo. Não alterar LEMA nem grafia das palavras mantidas.


## 1.16. Nome Própio Lugar — aplicar 7 correções aprovadas


In [17]:
# Lista completa das decisões aprovadas — Nome Própio Lugar
correcoes_nome_proprio_lugar = [{'LINHA_EXCEL': 235,
  'PALAVRA': 'ajudante',
  'LEMA': 'ajudá-los',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Nome Comun Comum Singular',
  'JUSTIFICATIVA': 'Admite substantivo comum de dois gêneros: pessoa que ajuda. A proposta '
                   'considera o uso lexical comum, não eventual nome de local.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 641,
  'PALAVRA': 'atacá-los',
  'LEMA': 'atacá-los',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de atacar com pronome enclítico los; não é nome geográfico. '
                   'Preservar a palavra inteira e o LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 891,
  'PALAVRA': 'belos',
  'LEMA': 'belos',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Plural',
  'JUSTIFICATIVA': 'Admite plural masculino de belo. A proposta considera o adjetivo; se houver '
                   'referente geográfico próprio comprovado no corpus, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3169,
  'PALAVRA': 'estremilique',
  'LEMA': 'estremilique',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino que designa tremor ou abalo; não há leitura '
                   'geográfica demonstrada na palavra isolada.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3287,
  'PALAVRA': 'familia',
  'LEMA': 'familia',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Na leitura família, admite substantivo feminino singular que designa grupo '
                   'familiar. Preservar a grafia e o LEMA.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5068,
  'PALAVRA': 'natalina',
  'LEMA': 'natalina',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Admite feminino singular de natalino, relativo ao Natal, além de nome pessoal. '
                   'A proposta adjetival é uma análise possível; se houver lugar com esse nome '
                   'comprovado no corpus, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6296,
  'PALAVRA': 'reencontrá-los',
  'LEMA': 'reencontrá-los',
  'CATMORF': 'Nome Própio Lugar',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Infinitivo de reencontrar com pronome enclítico los; preservar palavra inteira '
                   'e LEMA.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_nome_proprio_lugar)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_nome_proprio_lugar)
print("Grupo: Nome Própio Lugar")
print("Correções aprovadas:", len(correcoes_nome_proprio_lugar))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_nome_proprio_lugar) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_nome_proprio_lugar)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Própio Lugar
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
ajudante,ajudá-los,Nome Própio Lugar,Nome Comun Comum Singular,Aprovada
atacá-los,atacá-los,Nome Própio Lugar,Verbo Principal Infinitivo,Aprovada
belos,belos,Nome Própio Lugar,Adjetivo Qualificativo Masculino Plural,Aprovada
estremilique,estremilique,Nome Própio Lugar,Nome Comun Masculino Singular,Aprovada
familia,familia,Nome Própio Lugar,Nome Comun Feminino Singular,Aprovada
natalina,natalina,Nome Própio Lugar,Adjetivo Qualificativo Feminino Singular,Aprovada
reencontrá-los,reencontrá-los,Nome Própio Lugar,Verbo Principal Infinitivo,Aprovada


## 1.17. Nome Comun Comum Singular — aplicar 7 correções aprovadas

In [18]:
# Lista completa das decisões aprovadas — Nome Comun Comum Singular
correcoes_nome_comum_comum_singular = [{'LINHA_EXCEL': 281,
  'PALAVRA': 'alimentos',
  'LEMA': 'alícia',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Plural',
  'JUSTIFICATIVA': 'Plural masculino de alimento; não substantivo singular comum de dois gêneros. '
                   'Preservar o lema alícia.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3250,
  'PALAVRA': 'fabrícia',
  'LEMA': 'fabrícia',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite nome pessoal Fabrícia; não substantivo comum de dois gêneros.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3735,
  'PALAVRA': 'geral',
  'LEMA': 'geral',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Admite adjetivo comum de dois gêneros. Também tem usos nominais masculinos ou '
                   'femininos com sentidos distintos; a proposta adjetival é uma análise possível. '
                   'Se houver uso nominal comum de dois gêneros comprovado no corpus, manter a '
                   'categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 3777,
  'PALAVRA': 'gorila',
  'LEMA': 'gorila',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino, inclusive na leitura zoológica epicena: o gorila macho '
                   'e o gorila fêmea. Gênero gramatical não equivale ao sexo do animal.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4467,
  'PALAVRA': 'luar',
  'LEMA': 'luar',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa a luz da Lua; não comum de dois '
                   'gêneros.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 6750,
  'PALAVRA': 'silvestre',
  'LEMA': 'silvestre',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Comum Singular',
  'JUSTIFICATIVA': 'Admite adjetivo comum de dois gêneros, relativo à selva ou que vive sem '
                   'domesticação; também pode ser nome pessoal. A proposta considera a leitura '
                   'lexical adjetival; manter se houver substantivação comum de dois gêneros '
                   'comprovada no contexto.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 7682,
  'PALAVRA': 'zero',
  'LEMA': 'zero',
  'CATMORF': 'Nome Comun Comum Singular',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Admite substantivo masculino singular: o zero. Também admite numeral e forma '
                   'de zerar; a proposta nominal é uma análise possível, sem preferência '
                   'obrigatória.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_nome_comum_comum_singular)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_nome_comum_comum_singular)
print("Grupo: Nome Comun Comum Singular")
print("Correções aprovadas:", len(correcoes_nome_comum_comum_singular))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_nome_comum_comum_singular) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_nome_comum_comum_singular)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Nome Comun Comum Singular
Correções aprovadas: 7
Aplicadas nesta execução: 0
Já estavam corrigidas: 7


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
alimentos,alícia,Nome Comun Comum Singular,Nome Comun Masculino Plural,Aprovada
fabrícia,fabrícia,Nome Comun Comum Singular,Nome Própio Pessoa,Aprovada
geral,geral,Nome Comun Comum Singular,Adjetivo Qualificativo Comum Singular,Aprovada
gorila,gorila,Nome Comun Comum Singular,Nome Comun Masculino Singular,Aprovada
luar,luar,Nome Comun Comum Singular,Nome Comun Masculino Singular,Aprovada
silvestre,silvestre,Nome Comun Comum Singular,Adjetivo Qualificativo Comum Singular,Aprovada
zero,zero,Nome Comun Comum Singular,Nome Comun Masculino Singular,Aprovada


## 1.18. Quantidade — remoções aprovadas

Remover escarlete, fazero, humhum, hunhum, huum, huuum, ormitominco, tadum, tetete, tigum, vemeia, xove, deixove, lusarenta pela lista da seção 2. Não há correções de categoria neste grupo. Não alterar LEMA nem grafia das palavras mantidas.


## 1.19. Verbo Principal Indicativo Presente Primeira Pessoa Plural — remoções aprovadas

Remover ormitomimos, outumos pela lista da seção 2. Não há correções de categoria neste grupo. Não alterar LEMA nem grafia das palavras mantidas.


## 1.20. Verbo Principal Particípio Plural Masculino — aplicar 1 correção aprovada

In [19]:
# Lista completa das decisões aprovadas — Verbo Principal Particípio Plural Masculino
correcoes_participio_plural_masculino = [{'LINHA_EXCEL': 249,
  'PALAVRA': 'album',
  'LEMA': 'alaranjar',
  'CATMORF': 'Verbo Principal Particípio Plural Masculino',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Na leitura álbum, admite substantivo masculino singular. Não é particípio '
                   'masculino plural; preservar grafia e LEMA, inclusive o lema alaranjar.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_participio_plural_masculino)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_participio_plural_masculino)
print("Grupo: Verbo Principal Particípio Plural Masculino")
print("Correções aprovadas:", len(correcoes_participio_plural_masculino))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_participio_plural_masculino) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_participio_plural_masculino)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Particípio Plural Masculino
Correções aprovadas: 1
Aplicadas nesta execução: 0
Já estavam corrigidas: 1


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
album,alaranjar,Verbo Principal Particípio Plural Masculino,Nome Comun Masculino Singular,Aprovada


## 1.21. Adjetivo Qualificativo Comum Plural — aplicar 1 correção aprovada

Corrigir a grafia aprovada: doris → dóris. Não alterar LEMA.


In [20]:
# Lista completa das decisões aprovadas — Adjetivo Qualificativo Comum Plural
correcoes_adjetivo_comum_plural = [{'LINHA_EXCEL': 2566,
  'PALAVRA': 'doris',
  'LEMA': 'doris',
  'CATMORF': 'Adjetivo Qualificativo Comum Plural',
  'CATEGORIA_SUGERIDA': 'Nome Própio Pessoa',
  'JUSTIFICATIVA': 'Admite o nome pessoal Doris ou Dóris, não adjetivo comum plural. Há outras '
                   'leituras nominais possíveis; a proposta pessoal é uma análise possível, sem '
                   'preferência obrigatória.',
  'DECISAO': 'Aprovada',
  'PALAVRA_CORRIGIDA': 'dóris'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adjetivo_comum_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adjetivo_comum_plural)
print("Grupo: Adjetivo Qualificativo Comum Plural")
print("Correções aprovadas:", len(correcoes_adjetivo_comum_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adjetivo_comum_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adjetivo_comum_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])

# Correção de grafia aprovada, sem alterar LEMA ou outras colunas.
grafias_grupo = {'doris': 'dóris'}
antes_grafia = resultado.copy(deep=True)
for palavra_anterior, palavra_corrigida in grafias_grupo.items():
    mascara = resultado['PALAVRA'].isin([palavra_anterior, palavra_corrigida])
    assert mascara.sum() == 1, f'Palavra ausente ou duplicada: {palavra_anterior}'
    indice = resultado.index[mascara][0]
    if resultado.at[indice, 'PALAVRA'] == palavra_anterior:
        resultado.at[indice, 'PALAVRA'] = palavra_corrigida
        alteracoes_grafia.append({'PALAVRA_ANTERIOR': palavra_anterior, 'PALAVRA_CORRIGIDA': palavra_corrigida})
correcoes_grafia_aprovadas.update(grafias_grupo)
pd.testing.assert_frame_equal(antes_grafia.drop(columns='PALAVRA'), resultado.drop(columns='PALAVRA'))
print('Grafias aprovadas neste grupo:', grafias_grupo)


Grupo: Adjetivo Qualificativo Comum Plural
Correções aprovadas: 1
Aplicadas nesta execução: 1
Já estavam corrigidas: 0
Grafias aprovadas neste grupo: {'doris': 'dóris'}


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
doris,doris,Adjetivo Qualificativo Comum Plural,Nome Própio Pessoa,Aprovada


## 1.22. Adjetivo Qualificativo Feminino Plural — aplicar 2 correções aprovadas

Não alterar LEMA.


In [21]:
# Lista completa das decisões aprovadas — Adjetivo Qualificativo Feminino Plural
correcoes_adjetivo_feminino_plural = [{'LINHA_EXCEL': 355,
  'PALAVRA': 'amiga',
  'LEMA': 'americano',
  'CATMORF': 'Adjetivo Qualificativo Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Feminino Singular',
  'JUSTIFICATIVA': 'Admite adjetivo feminino singular; o plural seria amigas. Também há leitura '
                   'nominal, sem preferência obrigatória. Preservar o lema americano.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 416,
  'PALAVRA': 'anunciando',
  'LEMA': 'antigo',
  'CATMORF': 'Adjetivo Qualificativo Feminino Plural',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Gerúndio',
  'JUSTIFICATIVA': 'Gerúndio de anunciar, não adjetivo feminino plural. Preservar o lema antigo.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_adjetivo_feminino_plural)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_adjetivo_feminino_plural)
print("Grupo: Adjetivo Qualificativo Feminino Plural")
print("Correções aprovadas:", len(correcoes_adjetivo_feminino_plural))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_adjetivo_feminino_plural) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_adjetivo_feminino_plural)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Adjetivo Qualificativo Feminino Plural
Correções aprovadas: 2
Aplicadas nesta execução: 2
Já estavam corrigidas: 0


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
amiga,americano,Adjetivo Qualificativo Feminino Plural,Adjetivo Qualificativo Feminino Singular,Aprovada
anunciando,antigo,Adjetivo Qualificativo Feminino Plural,Verbo Principal Gerúndio,Aprovada


## 1.23. Verbo Principal Subjuntivo Presente Primeira Pessoa Singular — remoção aprovada

Remover carrandache pela lista da seção 2. Não há correções de categoria neste grupo.


## 1.24. Verbo Principal Particípio Plural Feminino — remoções aprovadas

Remover dosdas e fdas pela lista da seção 2. Não há correções de categoria neste grupo.


## 1.25. Nome Própio Organização — aplicar 4 correções aprovadas

Remover começarr, compraa, estu, grufalo, paraos, pararao, plaplapla, tirica, bolsona, coms, conse, estinamar, filica, pras pela lista da seção 2. Corrigir a grafia aprovada: minasgerais → minas gerais. Não alterar LEMA.


In [22]:
# Lista completa das decisões aprovadas — Nome Própio Organização
correcoes_nome_proprio_organizacao = [{'LINHA_EXCEL': 3161,
  'PALAVRA': 'estrela-do-mar',
  'LEMA': 'estrela-do-mar',
  'CATMORF': 'Nome Própio Organização',
  'CATEGORIA_SUGERIDA': 'Nome Comun Feminino Singular',
  'JUSTIFICATIVA': 'Admite substantivo comum feminino singular que designa o animal. A proposta '
                   'considera o uso lexical; se houver organização com esse nome comprovada no '
                   'corpus, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4786,
  'PALAVRA': 'minasgerais',
  'LEMA': 'minasgerais',
  'CATMORF': 'Nome Própio Organização',
  'CATEGORIA_SUGERIDA': 'Nome Própio Lugar',
  'JUSTIFICATIVA': 'Na leitura Minas Gerais, admite nome geográfico do estado brasileiro. A '
                   'proposta considera esse referente; se nomear uma organização no contexto, '
                   'manter a categoria atual.',
  'DECISAO': 'Aprovada',
  'PALAVRA_CORRIGIDA': 'minas gerais'},
 {'LINHA_EXCEL': 5357,
  'PALAVRA': 'paraibano',
  'LEMA': 'paraibano',
  'CATMORF': 'Nome Própio Organização',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Admite adjetivo masculino singular relativo à Paraíba. Também há leitura '
                   'nominal ou geográfica; a proposta adjetival é uma análise possível. Se '
                   'designar organização comprovada no corpus, manter a categoria atual.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 5385,
  'PALAVRA': 'participante',
  'LEMA': 'participante',
  'CATMORF': 'Nome Própio Organização',
  'CATEGORIA_SUGERIDA': 'Nome Comun Comum Singular',
  'JUSTIFICATIVA': 'Admite substantivo comum de dois gêneros: pessoa que participa. Também há '
                   'leitura adjetival; a proposta nominal é uma análise possível. Manter a '
                   'categoria atual se houver organização com esse nome comprovada no contexto.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_nome_proprio_organizacao)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_nome_proprio_organizacao)
print("Grupo: Nome Própio Organização")
print("Correções aprovadas:", len(correcoes_nome_proprio_organizacao))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_nome_proprio_organizacao) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_nome_proprio_organizacao)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])

# Correção de grafia aprovada, sem alterar LEMA ou outras colunas.
grafias_grupo = {'minasgerais': 'minas gerais'}
antes_grafia = resultado.copy(deep=True)
for palavra_anterior, palavra_corrigida in grafias_grupo.items():
    mascara = resultado['PALAVRA'].isin([palavra_anterior, palavra_corrigida])
    assert mascara.sum() == 1, f'Palavra ausente ou duplicada: {palavra_anterior}'
    indice = resultado.index[mascara][0]
    if resultado.at[indice, 'PALAVRA'] == palavra_anterior:
        resultado.at[indice, 'PALAVRA'] = palavra_corrigida
        alteracoes_grafia.append({'PALAVRA_ANTERIOR': palavra_anterior, 'PALAVRA_CORRIGIDA': palavra_corrigida})
correcoes_grafia_aprovadas.update(grafias_grupo)
pd.testing.assert_frame_equal(antes_grafia.drop(columns='PALAVRA'), resultado.drop(columns='PALAVRA'))
print('Grafias aprovadas neste grupo:', grafias_grupo)


Grupo: Nome Própio Organização
Correções aprovadas: 4
Aplicadas nesta execução: 4
Já estavam corrigidas: 0
Grafias aprovadas neste grupo: {'minasgerais': 'minas gerais'}


PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
estrela-do-mar,estrela-do-mar,Nome Própio Organização,Nome Comun Feminino Singular,Aprovada
minasgerais,minasgerais,Nome Própio Organização,Nome Própio Lugar,Aprovada
paraibano,paraibano,Nome Própio Organização,Adjetivo Qualificativo Masculino Singular,Aprovada
participante,participante,Nome Própio Organização,Nome Comun Comum Singular,Aprovada


## 2. Palavras identificadas como fora do escopo

In [23]:
palavras_remover = ['vamuos', 'ohmeudeus', 'sujerada', 'aiai', 'aiaiai', 'aimeudeus', 'alou', 'atchim', 'ei', 'pocotó', 'salabim', 'salimbim', 'simlabim', 'simsalabim', 'triim', 'xííí', 'estragda', 'liinda', 'aproximouse', 'disfarçouse', 'bixo', 'dodoí', 'dodí', 'hipopotámo', 'lapis', 'mosntro', 'piui', 'conisgo', 'tirouo', 'mostreio', 'ormito', 'robicho', 'bringuento', 'ciso', 'vestigo', 'saiubom', 'sobom', 'bembom', 'dexa', 'escredora', 'ninica', 'pricesa', 'rarara', 'rararara', 'rarararara', 'sugeira', 'manica', 'prara', 'ibirabuera', 'bumum', 'rinoce', 'otrem', 'hava', 'iiirrirria', 'irrrria', 'irrrrrria', 'puvava', 'ameli', 'atrave', 'provisiram', 'balnça', 'escarletina', 'scarletina', 'guilia', 'macdonalds', 'ponita', 'porcalina', 'psicina', 'visconde', 'xique', 'mareste', 'palpo', 'escarlete', 'fazero', 'humhum', 'hunhum', 'huum', 'huuum', 'ormitominco', 'tadum', 'tetete', 'tigum', 'vemeia', 'xove', 'deixove', 'lusarenta', 'ormitomimos', 'outumos', 'carrandache', 'começarr', 'compraa', 'estu', 'grufalo', 'paraos', 'pararao', 'plaplapla', 'tirica', 'dosdas', 'fdas', 'bolsona', 'coms', 'conse', 'estinamar', 'filica', 'pras']

total_antes = len(resultado)
mascara_remocao = resultado["PALAVRA"].isin(palavras_remover)
removidas = resultado.loc[mascara_remocao].copy()
nao_encontradas = sorted(set(palavras_remover) - set(removidas["PALAVRA"]))
resultado = resultado.loc[~mascara_remocao].reset_index(drop=True)
quantidade_removida = len(removidas)
assert total_antes - quantidade_removida == len(resultado)
print("Remoções previstas nesta execução:", quantidade_removida)
print("Total previsto na base:", len(resultado))
if nao_encontradas:
    print("Palavras já removidas ou não encontradas:", nao_encontradas)
display(removidas[["PALAVRA", "LEMA", "CATMORF"]])


Remoções previstas nesta execução: 17
Total previsto na base: 7671
Palavras já removidas ou não encontradas: ['aiai', 'aiaiai', 'aimeudeus', 'alou', 'ameli', 'aproximouse', 'atchim', 'atrave', 'balnça', 'bembom', 'bixo', 'bringuento', 'bumum', 'ciso', 'conisgo', 'deixove', 'dexa', 'disfarçouse', 'dodoí', 'dodí', 'ei', 'escarlete', 'escarletina', 'escredora', 'estragda', 'fazero', 'guilia', 'hava', 'hipopotámo', 'humhum', 'hunhum', 'huum', 'huuum', 'ibirabuera', 'iiirrirria', 'irrrria', 'irrrrrria', 'lapis', 'liinda', 'lusarenta', 'macdonalds', 'manica', 'mareste', 'mosntro', 'mostreio', 'ninica', 'ohmeudeus', 'ormito', 'ormitomimos', 'ormitominco', 'otrem', 'outumos', 'palpo', 'piui', 'pocotó', 'ponita', 'porcalina', 'prara', 'pricesa', 'provisiram', 'psicina', 'puvava', 'rarara', 'rararara', 'rarararara', 'rinoce', 'robicho', 'saiubom', 'salabim', 'salimbim', 'scarletina', 'simlabim', 'simsalabim', 'sobom', 'sugeira', 'sujerada', 'tadum', 'tetete', 'tigum', 'tirouo', 'triim', 'vamuos'

PALAVRA,LEMA,CATMORF
bolsona,bolsona,Nome Própio Organização
carrandache,carrandache,Verbo Principal Subjuntivo Presente Primeira Pessoa Singular
começarr,começarr,Nome Própio Organização
compraa,compraa,Nome Própio Organização
coms,coms,Nome Própio Organização
conse,conse,Nome Própio Organização
dosdas,dosdas,Verbo Principal Particípio Plural Feminino
estinamar,estinamar,Nome Própio Organização
estu,estu,Nome Própio Organização
fdas,fdas,Verbo Principal Particípio Plural Feminino


## 3. Salvamento e contagens

In [24]:
# Execute as células anteriores antes de salvar, incluindo a seção 2.
etapas_necessarias = {
    "arquivo": "carregamento da base",
    "base_original": "carregamento da base",
    "hash_entrada": "carregamento da base",
    "resultado": "preparação do resultado e correções",
    "alteracoes": "preparação do resultado e correções",
    "palavras_remover": "seção 2: palavras fora do escopo",
    "quantidade_removida": "seção 2: palavras fora do escopo",
    "correcoes_grafia_aprovadas": "preparação e grafias aprovadas",
    "alteracoes_grafia": "preparação e grafias aprovadas",
}
etapas_pendentes = sorted({
    etapa for variavel, etapa in etapas_necessarias.items()
    if variavel not in globals()
})
if etapas_pendentes:
    raise RuntimeError(
        "Antes de salvar, execute: " + "; ".join(etapas_pendentes)
        + ". Após reiniciar o kernel, execute o notebook de cima para baixo."
    )

assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada, (
    "A planilha mudou desde o carregamento. Reexecute o notebook de cima para baixo antes de salvar."
)
retidas = base_original.loc[~base_original["PALAVRA"].isin(palavras_remover)].reset_index(drop=True)
retidas["PALAVRA"] = retidas["PALAVRA"].replace(correcoes_grafia_aprovadas)
pd.testing.assert_frame_equal(retidas.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
if resultado.equals(base_original):
    verificada = base_original
    print("A base já está atualizada; nenhum novo salvamento foi necessário.")
else:
    from tempfile import TemporaryDirectory
    # A limpeza inclui arquivos de inspeção gerados pela ferramenta, mesmo em caso de erro.
    with TemporaryDirectory(prefix="temporarios_", dir=arquivo.resolve().parent) as pasta_temporaria:
        temporario = Path(pasta_temporaria) / "base_verificacao.xlsx"
        pacote = Path(pasta_temporaria) / "operacao.json"
        matriz = [list(resultado.columns)] + json.loads(resultado.to_json(orient="values", force_ascii=False))
        pacote.write_text(json.dumps({
            "source": str(arquivo.resolve()), "target": str(temporario.resolve()),
            "originalRows": len(base_original) + 1, "values": matriz
        }, ensure_ascii=False), encoding="utf-8")
        runtime = Path(os.environ.get("USERPROFILE", str(Path.home()))) / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
        node = runtime / "bin/node.exe"
        ambiente = os.environ.copy()
        ambiente["CODEX_NODE_PACKAGES"] = str(runtime / "node_modules")
        codigo_js = "\nimport fs from 'node:fs/promises';\nimport {createRequire} from 'node:module';\nimport {pathToFileURL} from 'node:url';\nconst require = createRequire(process.env.CODEX_NODE_PACKAGES + '/../package.json');\nconst {FileBlob, SpreadsheetFile} = await import(pathToFileURL(require.resolve('@oai/artifact-tool')).href);\nconst job=JSON.parse(await fs.readFile(process.argv[1],'utf8'));\nconst wb=await SpreadsheetFile.importXlsx(await FileBlob.load(job.source));\nconst sheet=wb.worksheets.getItem('Sheet1');\nsheet.getRange('A1:H'+job.originalRows).clear({applyTo:'contents'});\nsheet.getRange('A1:H'+job.values.length).values=job.values;\nwb.recalculate();\nconsole.log((await wb.inspect({kind:'region',sheetId:'Sheet1',range:'A1:C5',maxChars:1000})).ndjson);\nconst out=await SpreadsheetFile.exportXlsx(wb);\nawait out.save(job.target);\n"
        processo = subprocess.run([str(node), "--input-type=module", "-e", codigo_js, str(pacote.resolve())],
                                 env=ambiente, capture_output=True, text=True, encoding="utf-8")
        if processo.returncode:
            raise RuntimeError(processo.stderr or processo.stdout)
        verificada = pd.read_excel(temporario)
        pd.testing.assert_frame_equal(resultado, verificada, check_dtype=False)
        assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
        if not resultado.equals(base_original):
            os.replace(temporario, arquivo)
print("Correções aplicadas nesta execução:", len(alteracoes))
print("Grafias corrigidas nesta execução:", len(alteracoes_grafia))
print("Remoções nesta execução:", quantidade_removida)
print("Total atual na base:", len(verificada))
print("Lemas e demais colunas das linhas mantidas: preservados, exceto categorias e grafias aprovadas.")


Correções aplicadas nesta execução: 7
Grafias corrigidas nesta execução: 2
Remoções nesta execução: 17
Total atual na base: 7671
Lemas e demais colunas das linhas mantidas: preservados, exceto categorias e grafias aprovadas.
